<a href="https://colab.research.google.com/github/aimeerim1/thesis/blob/main/SVM_ClassicMlModels(Mental_health).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv("Combined Data.csv")
print(df.shape)
print(df.head())
print(df.columns)

(15900, 3)
   Unnamed: 0                                          statement   status
0           0                                         oh my gosh  Anxiety
1           1  trouble sleeping, confused mind, restless hear...  Anxiety
2           2  All wrong, back off dear, forward doubt. Stay ...  Anxiety
3           3  I've shifted my focus to something else but I'...  Anxiety
4           4  I'm restless and restless, it's been a month n...  Anxiety
Index(['Unnamed: 0', 'statement', 'status'], dtype='object')


In [4]:
import pandas as pd

df = pd.read_csv("Combined Data.csv")

# Gereksiz sütunu sil
df = df.drop(columns=["Unnamed: 0"])

# Etiket dağılımına bak
print(df["status"].value_counts())

# Boş değer var mı?
print(df.isnull().sum())

status
Normal                  16351
Depression              15404
Suicidal                10653
Anxiety                  3888
Bipolar                  2877
Stress                   2669
Personality disorder     1201
Name: count, dtype: int64
statement    362
status         0
dtype: int64


In [5]:
import pandas as pd
import re

# Boş satırları sil
df = df.dropna()

# Binary etiket oluştur
df["label"] = df["status"].apply(lambda x: 0 if x == "Normal" else 1)

# Metin temizleme fonksiyonu
def clean_text(text):
    text = text.lower()                          # küçük harf
    text = re.sub(r"http\S+", "", text)          # linkleri sil
    text = re.sub(r"[^a-zA-Z\s]", "", text)     # özel karakter sil
    text = text.strip()                          # baştaki/sondaki boşluk
    return text

df["clean_statement"] = df["statement"].apply(clean_text)

# Kontrol
print(df["label"].value_counts())
print(df[["statement", "clean_statement", "label"]].head())

label
1    36338
0    16343
Name: count, dtype: int64
                                           statement  \
0                                         oh my gosh   
1  trouble sleeping, confused mind, restless hear...   
2  All wrong, back off dear, forward doubt. Stay ...   
3  I've shifted my focus to something else but I'...   
4  I'm restless and restless, it's been a month n...   

                                     clean_statement  label  
0                                         oh my gosh      1  
1  trouble sleeping confused mind restless heart ...      1  
2  all wrong back off dear forward doubt stay in ...      1  
3  ive shifted my focus to something else but im ...      1  
4  im restless and restless its been a month now ...      1  


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_statement"],
    df["label"],
    test_size=0.2,
    random_state=42
)

# TF-IDF
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# SVM модель (Linear SVM)
svm_model = LinearSVC()

# Обучение
svm_model.fit(X_train_tfidf, y_train)

# Предсказание классов
y_pred = svm_model.predict(X_test_tfidf)

# 🔥 SVM НЕ даёт predict_proba напрямую
# поэтому оборачиваем его в CalibratedClassifierCV
calibrated_svm = CalibratedClassifierCV(svm_model)
calibrated_svm.fit(X_train_tfidf, y_train)
y_prob = calibrated_svm.predict_proba(X_test_tfidf)[:, 1]

# Метрики
print("Accuracy :", accuracy_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))
print("\n", classification_report(y_test, y_pred))

Accuracy : 0.949890860776312
F1 Score : 0.9634197034778994
ROC-AUC  : 0.9873431912943683

               precision    recall  f1-score   support

           0       0.92      0.92      0.92      3308
           1       0.97      0.96      0.96      7229

    accuracy                           0.95     10537
   macro avg       0.94      0.94      0.94     10537
weighted avg       0.95      0.95      0.95     10537

